# Daisyworld — комплексная динамика

## Введение

В этом документе мы построим комплексный график изменения числа
маргариток, температуры и солнечной светимости в зависимости от
модельного времени. Это позволит нам наблюдать за взаимосвязью между
биологическими и физическими процессами в модели.

## Подготовка окружения

Загружаем необходимые пакеты и активируем проект:

In [1]:
using DrWatson
@quickactivate "project"
using Agents
using DataFrames
using Plots
include(srcdir("daisyworld.jl"))
using CairoMakie

## Определение функций для сбора данных

Определяем функции-предикаты для разделения маргариток по видам:

In [2]:
black(a) = a.breed == :black
white(a) = a.breed == :white

white (generic function with 1 method)

Настраиваем сбор данных о количестве агентов каждого вида:

In [3]:
adata = [(black, count), (white, count)]

2-element Vector{Tuple{Function, typeof(count)}}:
 (Main.black, count)
 (Main.white, count)

Определяем функцию для вычисления средней температуры по всей
поверхности:

In [4]:
temperature(model) = StatsBase.mean(model.temperature)

temperature (generic function with 1 method)

Настраиваем сбор данных о модели (температура и солнечная светимость):

In [5]:
mdata = [temperature, :solar_luminosity]

2-element Vector{Any}:
 temperature (generic function with 1 method)
 :solar_luminosity

## Инициализация и запуск модели

Создаём модель с начальной светимостью 1.0 и сценарием `:ramp`
(постепенное изменение светимости):

In [6]:
model = daisyworld(solar_luminosity = 1.0, scenario = :ramp)

StandardABM with 360 agents of type Daisy
 agents container: Dict
 space: GridSpaceSingle with size (30, 30), metric=chebyshev, periodic=true
 scheduler: fastest
 properties: temperature, solar_luminosity, max_age, surface_albedo, ratio, solar_change, tick, scenario

Запускаем симуляцию на 1000 шагов со сбором данных об агентах и модели:

In [7]:
agent_df, model_df = run!(model, 1000; adata = adata, mdata = mdata)

( 1001×3 DataFrame 
 Row │ time count_black count_white 
 │ Int64 Int64 Int64 
──────┼─────────────────────────────────
 1 │ 0 180 180
 2 │ 1 384 166
 3 │ 2 524 220
 4 │ 3 575 293
 5 │ 4 575 317
 6 │ 5 570 326
 7 │ 6 563 333
 8 │ 7 558 339
 9 │ 8 554 342
 10 │ 9 550 348
 11 │ 10 552 345
 ⋮ │ ⋮ ⋮ ⋮
 992 │ 991 0 896
 993 │ 992 0 900
 994 │ 993 0 898
 995 │ 994 0 898
 996 │ 995 0 898
 997 │ 996 0 893
 998 │ 997 0 892
 999 │ 998 0 895
 1000 │ 999 0 891
 1001 │ 1000 0 886
 980 rows omitted , 1001×3 DataFrame 
 Row │ time temperature solar_luminosity 
 │ Int64 Float64 Float64 
──────┼──────────────────────────────────────
 1 │ 0 16.9135 1.0
 2 │ 1 23.5664 1.0
 3 │ 2 28.4386 1.0
 4 │ 3 30.4729 1.0
 5 │ 4 30.9852 1.0
 6 │ 5 30.8891 1.0
 7 │ 6 30.6937 1.0
 8 │ 7 30.3329 1.0
 9 │ 8 29.9755 1.0
 10 │ 9 29.5864 1.0
 11 │ 10 29.406 1.0
 ⋮ │ ⋮ ⋮ ⋮
 992 │ 991 6.0069 1.375
 993 │ 992 5.06515 1.375
 994 │ 993 4.48266 1.375
 995 │ 994 4.09715 1.375
 996 │ 995 3.83991 1.375
 997 │ 996 3.80634 1.375
 998 │ 997 3.85057 1.375
 999 │ 998 3.76921 1.375
 1000 │ 999 3.78791 1.375
 1001 │ 1000 4.03948 1.375
 980 rows omitted )

## Построение комплексного графика

Создаём фигуру с тремя подграфиками:

In [8]:
figure = CairoMakie.Figure(size = (600, 600));

### График 1: Численность маргариток

In [9]:
ax1 = figure[1, 1] = Axis(figure, ylabel = "daisy count")
blackl = lines!(ax1, agent_df[!, :time], agent_df[!, :count_black], color = :red)
whitel = lines!(ax1, agent_df[!, :time], agent_df[!, :count_white], color = :blue)
figure[1, 2] = Legend(figure, [blackl, whitel], ["black", "white"])

Legend()

### График 2: Температура

In [10]:
ax2 = figure[2, 1] = Axis(figure, ylabel = "temperature")
lines!(ax2, model_df[!, :time], model_df[!, :temperature], color = :red)

Lines{Tuple{Vector{Point{2, Float64}}}}

### График 3: Солнечная светимость

In [11]:
ax3 = figure[3, 1] = Axis(figure, xlabel = "tick", ylabel = "luminosity")
lines!(ax3, model_df[!, :time], model_df[!, :solar_luminosity], color = :red)

Lines{Tuple{Vector{Point{2, Float64}}}}

### Настройка отображения

Скрываем подписи по оси X для верхних графиков:

In [12]:
for ax in (ax1, ax2); ax.xticklabelsvisible = false; end

## Сохранение результата

Сохраняем полученный комплексный график в директорию plots:

In [13]:
save(plotsdir("daisy_luminosity.png"), figure)

## Заключение

Мы построили комплексный график, показывающий взаимосвязь между
численностью маргариток, температурой поверхности и солнечной
светимостью. Сценарий `:ramp` демонстрирует, как модель реагирует на
постепенное изменение внешних условий.